# Día 4A · Reranking multilingüe

Compara BGE-M3 normal, HyDE, normal + reranking y HyDE + reranking sobre las mismas 20 preguntas y 345 chunks.

In [ ]:
!git clone -q https://github.com/nalpata/proyecto_ActividadGrado-Riesgos.git
%cd proyecto_ActividadGrado-Riesgos
!git checkout -q dia-04a-reranking-multilingue
!pip -q install pandas pyarrow sentence-transformers FlagEmbedding openai scikit-learn

## 1. Subir resultados de los Días 3
Selecciona simultáneamente `resultados_hyde_dia_03.zip` y `resultados_finales_hyde_dia_03.zip`. El primero contiene los documentos HyDE; el segundo contiene el checkpoint del juez v3.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, shutil, pandas as pd
uploaded=files.upload()
original=next(n for n in uploaded if n.startswith('resultados_hyde_dia_03'))
final=next(n for n in uploaded if n.startswith('resultados_finales_hyde_dia_03'))
day3=Path('/content/day3'); day3final=Path('/content/day3final')
day3.mkdir(exist_ok=True); day3final.mkdir(exist_ok=True)
with zipfile.ZipFile(original) as z:z.extractall(day3)
with zipfile.ZipFile(final) as z:z.extractall(day3final)
assert (day3/'hyde_documents.csv').exists(), 'Falta hyde_documents.csv en el ZIP original.'
assert (day3final/'judge_v3_checkpoint.jsonl').exists(), 'Falta el checkpoint del juez v3.'
print('Entradas validadas')

## 2. Ejecutar las cuatro configuraciones
Usa GPU T4. La descarga inicial de los modelos puede tardar varios minutos.

In [ ]:
!python -m src.retrieval.run_multilingual_reranking --embeddings data/processed/embedding/embeddings_bge_m3.parquet --questions data/evaluation/gold_questions.csv --hyde /content/day3/hyde_documents.csv --output-dir /content/dia4a --top-n 20 --top-k 5

## 3. Evaluar relevancia con el juez v3
El checkpoint reutiliza juicios anteriores y solo consulta pares nuevos.

In [ ]:
import os
from getpass import getpass
os.environ['OPENAI_API_KEY']=getpass('OPENAI_API_KEY: ')
shutil.copy(day3final/'judge_v3_checkpoint.jsonl',Path('/content/dia4a/judge_v3_checkpoint.jsonl'))

In [ ]:
!python -m src.retrieval.rejudge_hyde_results --results /content/dia4a/reranking_results.csv --questions data/evaluation/gold_questions.csv --output-dir /content/dia4a
metrics=pd.read_csv('/content/dia4a/metrics_by_method_v3.csv').sort_values('mrr',ascending=False)
display(metrics)

## 4. Descargar resultados

In [ ]:
result_zip=shutil.make_archive('/content/resultados_reranking_multilingue_dia_04a','zip','/content/dia4a')
files.download(result_zip)